In [2]:
import pandas as pd
from pathlib import Path

# Hardcoded project root (same as 01 & 02)
repo_root = Path(r"C:\Users\elois\OneDrive\Desktop\vitality-behaviour-analysis")

DATA_PATH = repo_root / "data" / "processed" / "daily_points.parquet"

daily_points_df = pd.read_parquet(DATA_PATH)

daily_points_df.head()


,date,daily_points
0,2014-12-25,0
1,2014-12-26,0
2,2014-12-27,0
3,2014-12-28,0
4,2014-12-30,0


In [3]:
weekly_df = (
    daily_points_df
    .set_index("date")
    .resample("W-FRI")
    .sum()
    .reset_index()
    .rename(columns={"daily_points": "weekly_points"})
)

In [4]:
weekly_df["ring_closed"] = (weekly_df["weekly_points"] >= 900).astype(int)

weekly_df["ring_closed"].value_counts(normalize=True)

ring_closed
0    0.945392
1    0.054608
Name: proportion, dtype: float64

In [6]:
daily = daily_points_df.copy()
daily["weekday"] = daily["date"].dt.day_name()
daily["week"] = daily["date"].dt.to_period("W-FRI")

wed_features = (
    daily
    .query("weekday in ['Saturday','Sunday','Monday','Tuesday','Wednesday']")
    .groupby("week")
    .agg(
        wed_points=("daily_points", "sum"),
        active_days=("daily_points", lambda s: (s > 0).sum()),
        max_day_points=("daily_points", "max"),
    )
    .reset_index()
)

wed_features.tail()

,week,wed_points,active_days,max_day_points
356,2026-02-07/2026-02-13,900,3,300
357,2026-02-14/2026-02-20,200,1,200
358,2026-02-21/2026-02-27,300,1,300
359,2026-02-28/2026-03-06,600,3,300
360,2026-03-07/2026-03-13,600,2,300


In [8]:
model_df = wed_features.merge(
    weekly_df.assign(week=weekly_df["date"].dt.to_period("W-FRI")),
    on="week",
    how="inner"
)

model_df.tail()

,week,wed_points,active_days,max_day_points,date,weekly_points,ring_closed
356,2026-02-07/2026-02-13,900,3,300,2026-02-13,1200,1
357,2026-02-14/2026-02-20,200,1,200,2026-02-20,300,0
358,2026-02-21/2026-02-27,300,1,300,2026-02-27,600,0
359,2026-02-28/2026-03-06,600,3,300,2026-03-06,600,0
360,2026-03-07/2026-03-13,600,2,300,2026-03-13,600,0


In [9]:
# Can I close if I do two strong workouts?
model_df["feasible_2_days"] = (model_df["wed_points"] + 600 >= 900).astype(int)

baseline_accuracy = (
    model_df["feasible_2_days"] == model_df["ring_closed"]
).mean()

baseline_accuracy

0.7091412742382271

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

features = ["wed_points", "active_days", "max_day_points"]
X = model_df[features]
y = model_df["ring_closed"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, shuffle=False, test_size=0.25
)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

pipe.fit(X_train, y_train)

pipe.score(X_test, y_test)

0.8681318681318682

In [12]:
# --- Cell 8: Predicted probabilities ---

# Predict probability of closing the ring by Friday
model_df["predicted_prob"] = pipe.predict_proba(X)[:, 1]

# Optional: predicted class using default 0.5 threshold
model_df["predicted_label"] = (model_df["predicted_prob"] >= 0.5).astype(int)

# Inspect key columns
cols_to_view = [
    "week",
    "wed_points",
    "active_days",
    "max_day_points",
    "weekly_points",
    "ring_closed",
    "predicted_prob",
    "predicted_label",
]

model_df[cols_to_view].tail(10)


,week,wed_points,active_days,max_day_points,weekly_points,ring_closed,predicted_prob,predicted_label
351,2026-01-03/2026-01-09,300,1,300,500,0,0.113147,0
352,2026-01-10/2026-01-16,400,2,200,600,0,0.065209,0
353,2026-01-17/2026-01-23,600,2,300,1100,1,0.405266,0
354,2026-01-24/2026-01-30,400,2,300,600,0,0.140184,0
355,2026-01-31/2026-02-06,400,2,300,500,0,0.140184,0
356,2026-02-07/2026-02-13,900,3,300,1200,1,0.784460,1
357,2026-02-14/2026-02-20,200,1,200,300,0,0.026007,0
358,2026-02-21/2026-02-27,300,1,300,600,0,0.113147,0
359,2026-02-28/2026-03-06,600,3,300,600,0,0.298713,0
360,2026-03-07/2026-03-13,600,2,300,600,0,0.405266,0


In [13]:
# Save model outputs for evaluation & calibration
OUT_PATH = repo_root / "data" / "processed" / "weekly_predictions.parquet"
model_df.to_parquet(OUT_PATH)

print("Saved weekly predictions →", OUT_PATH)




Saved weekly predictions → C:\Users\elois\OneDrive\Desktop\vitality-behaviour-analysis\data\processed\weekly_predictions.parquet


In [14]:

model_df["predicted_prob"].describe()



count    361.000000
mean       0.094111
std        0.173377
min        0.001867
25%        0.001867
50%        0.001867
75%        0.113147
max        0.980689
Name: predicted_prob, dtype: float64

In [15]:
model_df.groupby("ring_closed")["predicted_prob"].mean()

ring_closed
0    0.058511
1    0.460125
Name: predicted_prob, dtype: float64